# LeetCode 2363 — Merge Similar Items bằng Python

Link đề: https://leetcode.com/problems/merge-similar-items/ · Độ khó: **Easy** · Chủ đề: Array, Hash Table, Sorting, Ordered Set

Skeleton LeetCode:

```python
class Solution:
    def mergeSimilarItems(self, items1: List[List[int]], items2: List[List[int]]) -> List[List[int]]:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu bài toán **gộp hai danh sách cặp `[value, weight]`**: cùng `value` thì cộng `weight`, rồi sắp xếp theo `value` tăng dần.
2. Dùng `dict`, `collections.Counter` (phép cộng khi cập nhật), `dict.get(key, 0)` và `sorted(d.items())` để gộp theo khóa.
3. Biết **mảng đếm theo chỉ số giá trị** (vì `value ≤ 1000`) cho kết quả **tự động đã sắp xếp** mà không cần `sort`.
4. Viết 5 cách: dict + sắp xếp, `Counter`, mảng 1001 ô, sắp xếp hai danh sách rồi **hai con trỏ** (merge), và nối rồi `groupby`; đối chiếu bằng test ngẫu nhiên.
5. Tránh các lỗi: quên sắp xếp, ghi đè thay vì cộng, quên các giá trị chỉ có ở một danh sách, trả về danh sách `tuple` thay vì `list` `[value, weight]`.

## 2. Đề bài & ví dụ

Cho hai mảng 2 chiều `items1` và `items2`, mỗi phần tử là `[value, weight]`:

- `value` là **giá trị định danh** của món đồ (không phải số lượng!).
- `weight` là **trọng lượng** của món đồ đó.

Trong **mỗi mảng**, mọi `value` là **duy nhất** (không lặp). Hãy trả về mảng `ret` gồm các cặp `[value, total_weight]`, trong đó với mỗi `value` có trong **ít nhất một** mảng, `total_weight` là **tổng trọng lượng** của nó ở cả hai mảng (thiếu ở mảng nào thì coi như `0`). `ret` phải **sắp xếp tăng dần theo `value`**.

| Ví dụ | items1 | items2 | Output |
|---|---|---|---|
| 1 | `[[1,1],[4,5],[3,8]]` | `[[3,1],[1,5]]` | `[[1,6],[3,9],[4,5]]` |
| 2 | `[[1,1],[3,2],[2,3]]` | `[[2,1],[3,2],[1,3]]` | `[[1,4],[2,4],[3,4]]` |
| 3 | `[[1,3],[2,2]]` | `[[7,1],[2,2],[1,4]]` | `[[1,7],[2,4],[7,1]]` |

Giải thích ví dụ 1: `value=1` có trọng lượng `1+5=6`; `value=3`: `8+1=9`; `value=4` chỉ có ở `items1`: `5`.

Constraints: `1 <= len(items1), len(items2) <= 1000`; `items[i].length == 2`; `1 <= value, weight <= 1000`; `value` duy nhất trong mỗi mảng.

In [1]:
examples = [
    ([[1, 1], [4, 5], [3, 8]], [[3, 1], [1, 5]], [[1, 6], [3, 9], [4, 5]]),
    ([[1, 1], [3, 2], [2, 3]], [[2, 1], [3, 2], [1, 3]], [[1, 4], [2, 4], [3, 4]]),
    ([[1, 3], [2, 2]], [[7, 1], [2, 2], [1, 4]], [[1, 7], [2, 4], [7, 1]]),
]
for a, b, expected in examples:
    total = {}
    for v, w in a + b:
        total[v] = total.get(v, 0) + w
    print(f"{a} + {b}\n   -> tổng theo value: {total} -> {sorted(total.items())} ; mong đợi {expected}")

[[1, 1], [4, 5], [3, 8]] + [[3, 1], [1, 5]]
   -> tổng theo value: {1: 6, 4: 5, 3: 9} -> [(1, 6), (3, 9), (4, 5)] ; mong đợi [[1, 6], [3, 9], [4, 5]]
[[1, 1], [3, 2], [2, 3]] + [[2, 1], [3, 2], [1, 3]]
   -> tổng theo value: {1: 4, 3: 4, 2: 4} -> [(1, 4), (2, 4), (3, 4)] ; mong đợi [[1, 4], [2, 4], [3, 4]]
[[1, 3], [2, 2]] + [[7, 1], [2, 2], [1, 4]]
   -> tổng theo value: {1: 7, 2: 4, 7: 1} -> [(1, 7), (2, 4), (7, 1)] ; mong đợi [[1, 7], [2, 4], [7, 1]]


## 3. Kiến thức nền

### 3.1. Danh sách các cặp và "giải nén" `for v, w in items`

Mỗi phần tử `[value, weight]` có thể giải nén ngay trong vòng `for`: `for v, w in items:`.

In [2]:
items = [[1, 1], [4, 5], [3, 8]]
for v, w in items:
    print("value", v, "weight", w)
print(items[0][0], items[0][1], len(items))

value 1 weight 1
value 4 weight 5
value 3 weight 8
1 1 3


### 3.2. `dict` và `dict.get(key, default)`

`d[key] = d.get(key, 0) + w` là cách chuẩn để **cộng dồn** theo khóa: nếu `key` chưa có thì `get` cho `0`. Đừng viết `d[key] += w` khi `key` có thể chưa tồn tại (sẽ `KeyError`).

In [3]:
d = {}
for v, w in [[1, 1], [3, 2], [1, 5]]:
    d[v] = d.get(v, 0) + w
print(d)
try:
    e = {}
    e[1] += 1
except KeyError as err:
    print("KeyError:", err)

{1: 6, 3: 2}
KeyError: 1


### 3.3. `collections.Counter` và `defaultdict`

- `Counter()`: từ điển đếm; khóa vắng mặt cho `0`; `c[key] += w` hợp lệ ngay.
- `defaultdict(int)`: từ điển có giá trị mặc định `0`.
Cả hai đều tránh `KeyError` khi cộng dồn.

In [4]:
from collections import Counter, defaultdict
c = Counter()
for v, w in [[1, 1], [3, 2], [1, 5]]:
    c[v] += w
print(c, c[99])
dd = defaultdict(int)
dd[7] += 4
print(dict(dd))

Counter({1: 6, 3: 2}) 0
{7: 4}


### 3.4. `sorted(d.items())` và danh sách cặp

`d.items()` cho các cặp `(khóa, giá trị)`. `sorted(...)` sắp xếp theo phần tử đầu tiên của cặp (khóa) trước. Kết quả là danh sách các **tuple**; bài yêu cầu danh sách các **list** nên đổi bằng `[[k, v] for k, v in ...]`.

In [5]:
d = {4: 5, 1: 6, 3: 9}
print(sorted(d.items()))
print([[k, v] for k, v in sorted(d.items())])
print([1, 6] == (1, 6), [1, 6] == list((1, 6)))

[(1, 6), (3, 9), (4, 5)]
[[1, 6], [3, 9], [4, 5]]
False True


### 3.5. Mảng đếm theo chỉ số

Nếu giá trị `value` nằm trong khoảng nhỏ (`1..1000`), ta dùng mảng `weight_of = [0] * 1001`; `weight_of[v] += w`. Duyệt `v` từ `1` đến `1000` cho các giá trị **theo thứ tự tăng dần** mà **không cần sắp xếp**.

In [6]:
weight_of = [0] * 6
for v, w in [[1, 1], [4, 5], [3, 8], [3, 1], [1, 5]]:
    weight_of[v] += w
print(weight_of)
print([[v, weight_of[v]] for v in range(len(weight_of)) if weight_of[v] > 0])

[0, 6, 0, 9, 5, 0]
[[1, 6], [3, 9], [4, 5]]


### 3.6. Hai con trỏ trên hai danh sách đã sắp xếp (merge)

Giống bước trộn của merge sort: hai con trỏ `i`, `j`; so sánh `a[i][0]` với `b[j][0]`; lấy giá trị nhỏ hơn; nếu bằng nhau thì **cộng** hai trọng lượng và tiến cả hai.

In [7]:
a = [1, 3, 4]
b = [1, 2, 4, 5]
i = j = 0
out = []
while i < len(a) and j < len(b):
    if a[i] == b[j]:
        out.append(("cả hai", a[i])); i += 1; j += 1
    elif a[i] < b[j]:
        out.append(("chỉ a", a[i])); i += 1
    else:
        out.append(("chỉ b", b[j])); j += 1
out += [("chỉ a", x) for x in a[i:]] + [("chỉ b", x) for x in b[j:]]
print(out)

[('cả hai', 1), ('chỉ b', 2), ('chỉ a', 3), ('cả hai', 4), ('chỉ b', 5)]


### 3.7. `itertools.groupby` trên danh sách đã sắp xếp theo khóa

`groupby(sorted_items, key=lambda t: t[0])` gom các cặp **cùng `value` liền kề** thành một nhóm; ta cộng `weight` trong nhóm. Cần **sắp xếp trước** theo cùng khóa.

In [8]:
from itertools import groupby
data = sorted([[1, 1], [4, 5], [3, 8], [3, 1], [1, 5]])
print(data)
print([[k, sum(w for _, w in g)] for k, g in groupby(data, key=lambda t: t[0])])

[[1, 1], [1, 5], [3, 1], [3, 8], [4, 5]]
[[1, 6], [3, 9], [4, 5]]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Giống **gộp hai hóa đơn**: cùng tên mặt hàng (`value`) thì cộng số lượng/trọng lượng (`weight`); mặt hàng chỉ ở một hóa đơn thì giữ nguyên; rồi in ra theo thứ tự tăng dần của mã mặt hàng.

Ví dụ 1: `items1 = [[1,1],[4,5],[3,8]]`, `items2 = [[3,1],[1,5]]`.

| value | ở `items1` | ở `items2` | tổng |
|---|---|---|---|
| 1 | 1 | 5 | **6** |
| 3 | 8 | 1 | **9** |
| 4 | 5 | (không có) | **5** |

Sắp xếp theo `value`: `[[1,6],[3,9],[4,5]]`. Đây chính là đáp án.

Vậy: **cộng dồn theo `value` (từ điển), rồi sắp xếp theo `value`**.

In [9]:
items1 = [[1, 1], [4, 5], [3, 8]]
items2 = [[3, 1], [1, 5]]
total = {}
for source, items in (("items1", items1), ("items2", items2)):
    for v, w in items:
        total[v] = total.get(v, 0) + w
        print(f"{source}: [{v},{w}] -> total[{v}] = {total[v]}")
print("sắp xếp:", [[v, total[v]] for v in sorted(total)])

items1: [1,1] -> total[1] = 1
items1: [4,5] -> total[4] = 5
items1: [3,8] -> total[3] = 8
items2: [3,1] -> total[3] = 9
items2: [1,5] -> total[1] = 6
sắp xếp: [[1, 6], [3, 9], [4, 5]]


## 5. Cách 1 — Brute Force / baseline: với mỗi `value` tìm trong cả hai danh sách

Không dùng từ điển: gom **tập các `value`** có mặt, rồi với mỗi `value` **quét** cả hai danh sách để cộng trọng lượng. Dễ hiểu nhưng quét lặp lại nhiều lần.

In [10]:
def merge_brute(items1, items2):
    values = sorted({v for v, _ in items1} | {v for v, _ in items2})
    result = []
    for v in values:
        total = 0
        for value, weight in items1:
            if value == v:
                total += weight
        for value, weight in items2:
            if value == v:
                total += weight
        result.append([v, total])
    return result

print(merge_brute([[1, 1], [4, 5], [3, 8]], [[3, 1], [1, 5]]))

[[1, 6], [3, 9], [4, 5]]


In [11]:
from typing import List

class SolutionBrute:
    def mergeSimilarItems(self, items1: List[List[int]], items2: List[List[int]]) -> List[List[int]]:
        values = sorted({v for v, _ in items1} | {v for v, _ in items2})
        result = []
        for v in values:
            total = 0
            for value, weight in items1:
                if value == v:
                    total += weight
            for value, weight in items2:
                if value == v:
                    total += weight
            result.append([v, total])
        return result

print(SolutionBrute().mergeSimilarItems([[1, 1], [4, 5], [3, 8]], [[3, 1], [1, 5]]))
print(SolutionBrute().mergeSimilarItems([[1, 3], [2, 2]], [[7, 1], [2, 2], [1, 4]]))

[[1, 6], [3, 9], [4, 5]]
[[1, 7], [2, 4], [7, 1]]


**Độ phức tạp:** `n = len(items1)`, `m = len(items2)`. Có tối đa `n + m` giá trị khác nhau, mỗi giá trị quét `n + m` phần tử → **Time `O((n + m)²)`**; **Space `O(n + m)`**. Với `n, m ≤ 1000` có tới `~4·10⁶` phép — vẫn chạy được nhưng lãng phí.

**Nút thắt:** việc **tìm lại** trọng lượng của một `value` bằng quét. Dùng `dict` để cộng dồn **một lượt**.

## 6. Ý tưởng thuật toán / bước đột phá

- Ánh xạ `value → tổng weight` là một **từ điển**; cộng dồn trong **một lượt** qua cả hai danh sách: `total[v] += w`.
- Vì mỗi `value` là duy nhất **trong mỗi mảng**, tổng của một `value` có tối đa hai số hạng; nhưng code tổng quát (cộng dồn) vẫn đúng.
- Yêu cầu **sắp xếp theo `value`**: `sorted(total.items())`, hoặc dùng **mảng đếm theo chỉ số** (đã tự có thứ tự), hoặc **sắp xếp hai danh sách rồi trộn** (như merge sort).
- Với `value ≤ 1000`, mảng đếm 1001 ô là cách `O(n + m + 1000)` không cần sắp xếp.

## 7. Các cách tiếp theo

### Cách 2 — `dict` cộng dồn + `sorted` (khuyến nghị khi nộp)

In [12]:
from typing import List

class SolutionDict:
    def mergeSimilarItems(self, items1: List[List[int]], items2: List[List[int]]) -> List[List[int]]:
        total = {}
        for value, weight in items1 + items2:
            total[value] = total.get(value, 0) + weight
        return [[value, total[value]] for value in sorted(total)]

print(SolutionDict().mergeSimilarItems([[1, 1], [4, 5], [3, 8]], [[3, 1], [1, 5]]))
print(SolutionDict().mergeSimilarItems([[1, 3], [2, 2]], [[7, 1], [2, 2], [1, 4]]))

[[1, 6], [3, 9], [4, 5]]
[[1, 7], [2, 4], [7, 1]]


**Độ phức tạp:** Time `O((n + m) log(n + m))` do sắp xếp các khóa, Space `O(n + m)`. (Dựng từ điển `O(n + m)`.)

### Cách 3 — `Counter` với phép cộng

`Counter` cho phép `c[v] += w` ngay cả khi `v` chưa có. Ngoài ra `Counter(dict1) + Counter(dict2)` cộng hai từ điển; ta dùng cú pháp đơn giản nhất.

In [13]:
from collections import Counter
from typing import List

class SolutionCounter:
    def mergeSimilarItems(self, items1: List[List[int]], items2: List[List[int]]) -> List[List[int]]:
        total = Counter()
        for value, weight in items1:
            total[value] += weight
        for value, weight in items2:
            total[value] += weight
        return sorted([v, w] for v, w in total.items())

print(SolutionCounter().mergeSimilarItems([[1, 1], [4, 5], [3, 8]], [[3, 1], [1, 5]]))
print(SolutionCounter().mergeSimilarItems([[1, 3], [2, 2]], [[7, 1], [2, 2], [1, 4]]))

[[1, 6], [3, 9], [4, 5]]
[[1, 7], [2, 4], [7, 1]]


**Độ phức tạp:** Time `O((n + m) log(n + m))`, Space `O(n + m)`. `sorted` trên các list `[v, w]` so sánh theo `v` trước (giá trị duy nhất nên không bao giờ xét đến `w`).

### Cách 4 — Mảng đếm 1001 ô (không cần `sort`)

Dùng `value ≤ 1000`. Duyệt `v` từ `1` đến `1000`: kết quả **tự động tăng dần**. Chú ý điều kiện "có mặt": `weight ≥ 1` nên tổng `> 0` khi `value` có mặt, do đó có thể lọc bằng `> 0`.

In [14]:
from typing import List

class SolutionBuckets:
    def mergeSimilarItems(self, items1: List[List[int]], items2: List[List[int]]) -> List[List[int]]:
        weight_of = [0] * 1001
        for value, weight in items1:
            weight_of[value] += weight
        for value, weight in items2:
            weight_of[value] += weight
        return [[v, weight_of[v]] for v in range(1, 1001) if weight_of[v] > 0]

print(SolutionBuckets().mergeSimilarItems([[1, 1], [4, 5], [3, 8]], [[3, 1], [1, 5]]))
print(SolutionBuckets().mergeSimilarItems([[1, 3], [2, 2]], [[7, 1], [2, 2], [1, 4]]))

[[1, 6], [3, 9], [4, 5]]
[[1, 7], [2, 4], [7, 1]]


**Độ phức tạp:** Time `O(n + m + V)` với `V = 1000`, Space `O(V)`. Không có `log` — thích hợp khi miền `value` nhỏ.

### Cách 5 — Sắp xếp hai danh sách rồi **hai con trỏ** (trộn)

Sắp xếp `items1` và `items2` theo `value`. Dùng `i`, `j`: bằng nhau thì cộng và tiến cả hai; `value` nhỏ hơn thì lấy riêng và tiến con trỏ đó. Cuối cùng nối phần còn lại. Cách này **không dùng từ điển**, bộ nhớ phụ chỉ cho kết quả.

In [15]:
from typing import List

class SolutionTwoPointers:
    def mergeSimilarItems(self, items1: List[List[int]], items2: List[List[int]]) -> List[List[int]]:
        a = sorted(items1)
        b = sorted(items2)
        i = j = 0
        result = []
        while i < len(a) and j < len(b):
            if a[i][0] == b[j][0]:
                result.append([a[i][0], a[i][1] + b[j][1]])
                i += 1
                j += 1
            elif a[i][0] < b[j][0]:
                result.append(a[i][:])
                i += 1
            else:
                result.append(b[j][:])
                j += 1
        result.extend(x[:] for x in a[i:])
        result.extend(x[:] for x in b[j:])
        return result

print(SolutionTwoPointers().mergeSimilarItems([[1, 1], [4, 5], [3, 8]], [[3, 1], [1, 5]]))
print(SolutionTwoPointers().mergeSimilarItems([[1, 3], [2, 2]], [[7, 1], [2, 2], [1, 4]]))

[[1, 6], [3, 9], [4, 5]]
[[1, 7], [2, 4], [7, 1]]


**Độ phức tạp:** Time `O(n log n + m log m)` do sắp xếp, phần trộn `O(n + m)`; Space `O(n + m)` cho kết quả (không cần từ điển). Ghi chú: `a[i][:]` sao chép để không trả về chính phần tử của đầu vào.

### Cách 6 (thêm) — Nối hai danh sách, sắp xếp, `groupby`

Nối `items1 + items2`, sắp xếp theo `value`, rồi dùng `groupby` để cộng các cặp cùng `value` liền kề. Gọn và dễ mở rộng cho trường hợp một `value` lặp nhiều lần.

In [16]:
from itertools import groupby
from typing import List

class SolutionGroupBy:
    def mergeSimilarItems(self, items1: List[List[int]], items2: List[List[int]]) -> List[List[int]]:
        merged = sorted(items1 + items2, key=lambda t: t[0])
        return [[value, sum(w for _, w in group)] for value, group in groupby(merged, key=lambda t: t[0])]

print(SolutionGroupBy().mergeSimilarItems([[1, 1], [4, 5], [3, 8]], [[3, 1], [1, 5]]))
print(SolutionGroupBy().mergeSimilarItems([[1, 3], [2, 2]], [[7, 1], [2, 2], [1, 4]]))

[[1, 6], [3, 9], [4, 5]]
[[1, 7], [2, 4], [7, 1]]


**Độ phức tạp:** Time `O((n + m) log(n + m))`, Space `O(n + m)`.

## 8. TRACE từng bước

Trace Cách 5 (hai con trỏ) với `items1 = [[1,1],[4,5],[3,8]]` (sau sắp xếp `[[1,1],[3,8],[4,5]]`) và `items2 = [[3,1],[1,5]]` (sau sắp xếp `[[1,5],[3,1]]`):

| bước | `i` | `j` | `a[i]` | `b[j]` | hành động | `result` |
|---|---|---|---|---|---|---|
| 1 | 0 | 0 | `[1,1]` | `[1,5]` | bằng nhau → cộng | `[[1,6]]` |
| 2 | 1 | 1 | `[3,8]` | `[3,1]` | bằng nhau → cộng | `[[1,6],[3,9]]` |
| — | 2 | 2 | hết `b` | | nối phần còn lại của `a` | `[[1,6],[3,9],[4,5]]` |

In [17]:
def trace_two_pointers(items1, items2):
    a, b = sorted(items1), sorted(items2)
    print("a =", a, "| b =", b)
    i = j = 0
    result = []
    step = 0
    while i < len(a) and j < len(b):
        step += 1
        if a[i][0] == b[j][0]:
            result.append([a[i][0], a[i][1] + b[j][1]])
            action = "bằng nhau -> cộng"
            i += 1; j += 1
        elif a[i][0] < b[j][0]:
            result.append(a[i][:])
            action = "a nhỏ hơn -> lấy a"
            i += 1
        else:
            result.append(b[j][:])
            action = "b nhỏ hơn -> lấy b"
            j += 1
        print(f"  bước {step}: {action:20} i={i} j={j} result={result}")
    result.extend(x[:] for x in a[i:])
    result.extend(x[:] for x in b[j:])
    print("  nối phần còn lại ->", result)
    return result

trace_two_pointers([[1, 1], [4, 5], [3, 8]], [[3, 1], [1, 5]])

a = [[1, 1], [3, 8], [4, 5]] | b = [[1, 5], [3, 1]]
  bước 1: bằng nhau -> cộng    i=1 j=1 result=[[1, 6]]
  bước 2: bằng nhau -> cộng    i=2 j=2 result=[[1, 6], [3, 9]]
  nối phần còn lại -> [[1, 6], [3, 9], [4, 5]]


[[1, 6], [3, 9], [4, 5]]

In [18]:
def trace_dict(items1, items2):
    total = {}
    print("items1 =", items1, "| items2 =", items2)
    for source, items in (("items1", items1), ("items2", items2)):
        for value, weight in items:
            total[value] = total.get(value, 0) + weight
            print(f"  {source}: cộng [{value},{weight}] -> total={total}")
    result = [[v, total[v]] for v in sorted(total)]
    print("  sắp xếp theo value ->", result)
    return result

trace_dict([[1, 3], [2, 2]], [[7, 1], [2, 2], [1, 4]])

items1 = [[1, 3], [2, 2]] | items2 = [[7, 1], [2, 2], [1, 4]]
  items1: cộng [1,3] -> total={1: 3}
  items1: cộng [2,2] -> total={1: 3, 2: 2}
  items2: cộng [7,1] -> total={1: 3, 2: 2, 7: 1}
  items2: cộng [2,2] -> total={1: 3, 2: 4, 7: 1}
  items2: cộng [1,4] -> total={1: 7, 2: 4, 7: 1}
  sắp xếp theo value -> [[1, 7], [2, 4], [7, 1]]


[[1, 7], [2, 4], [7, 1]]

## 9. Code nộp LeetCode

In [19]:
from typing import List

class Solution:
    def mergeSimilarItems(self, items1: List[List[int]], items2: List[List[int]]) -> List[List[int]]:
        total = {}
        for value, weight in items1 + items2:
            total[value] = total.get(value, 0) + weight
        return [[value, total[value]] for value in sorted(total)]

print(Solution().mergeSimilarItems([[1, 1], [4, 5], [3, 8]], [[3, 1], [1, 5]]))
print(Solution().mergeSimilarItems([[1, 1], [3, 2], [2, 3]], [[2, 1], [3, 2], [1, 3]]))
print(Solution().mergeSimilarItems([[1, 3], [2, 2]], [[7, 1], [2, 2], [1, 4]]))

[[1, 6], [3, 9], [4, 5]]
[[1, 4], [2, 4], [3, 4]]
[[1, 7], [2, 4], [7, 1]]


## 10. Vì sao thuật toán đúng

- **Tổng theo `value`:** sau khi duyệt hết `items1 + items2`, `total[v]` bằng tổng `weight` của **mọi** cặp có `value = v`; đây đúng là "tổng trọng lượng của `v` trong cả hai mảng" (thiếu ở một mảng thì không đóng góp, tương đương `0`).
- **Đủ giá trị:** mỗi `value` có ở ít nhất một mảng được cộng ít nhất một lần nên có mặt trong `total`; không có `value` nào ngoài hai mảng được thêm vào.
- **Thứ tự:** `sorted(total)` sắp xếp các khóa tăng dần; vì mỗi khóa duy nhất trong `total`, kết quả tăng chặt chẽ theo `value`.
- **Hai con trỏ:** invariant "mọi `value` nhỏ hơn `min(a[i][0], b[j][0])` đã được xử lý (kể cả cộng khi trùng)". Mỗi bước tiến ít nhất một con trỏ nên dừng sau `≤ n + m` bước; phần dư còn lại của một mảng không có đối ứng trong mảng kia nên được nối nguyên.
- **Mảng đếm:** duyệt `v = 1..1000` đảm bảo tăng dần; điều kiện `> 0` loại các `value` không có mặt (vì `weight ≥ 1` nên giá trị có mặt luôn có tổng dương).

## 11. Phân tích Time / Space Complexity

`n = len(items1)`, `m = len(items2)`, `V = 1000`.

| Cách | Time | Space |
|---|---|---|
| 1. Với mỗi `value` quét hai danh sách | `O((n+m)²)` | `O(n+m)` |
| 2. `dict` + `sorted` | `O((n+m) log(n+m))` | `O(n+m)` |
| 3. `Counter` + `sorted` | `O((n+m) log(n+m))` | `O(n+m)` |
| 4. Mảng 1001 ô | `O(n + m + V)` | `O(V)` |
| 5. Sắp xếp + hai con trỏ | `O(n log n + m log m)` | `O(n+m)` (kết quả) |
| 6. Nối + `groupby` | `O((n+m) log(n+m))` | `O(n+m)` |

Cận dưới là đọc `n + m` phần tử; cách 4 gần đạt `O(n + m)` vì miền giá trị nhỏ. Với `n, m ≤ 1000` mọi cách (trừ cách 1) chạy tức thời.

## 12. Bộ test

Ví dụ đề, edge case (một phần tử mỗi bên, không trùng `value` nào, trùng hoàn toàn, biên `1` và `1000`, thứ tự đầu vào ngược) và test ngẫu nhiên đối chiếu **tất cả cách**.

In [20]:
import random

solvers = [SolutionBrute(), SolutionDict(), SolutionCounter(), SolutionBuckets(), SolutionTwoPointers(), SolutionGroupBy(), Solution()]

fixed = [
    ([[1, 1], [4, 5], [3, 8]], [[3, 1], [1, 5]], [[1, 6], [3, 9], [4, 5]]),
    ([[1, 1], [3, 2], [2, 3]], [[2, 1], [3, 2], [1, 3]], [[1, 4], [2, 4], [3, 4]]),
    ([[1, 3], [2, 2]], [[7, 1], [2, 2], [1, 4]], [[1, 7], [2, 4], [7, 1]]),
    ([[5, 5]], [[5, 5]], [[5, 10]]),                          # một phần tử mỗi bên, trùng
    ([[1, 1]], [[2, 2]], [[1, 1], [2, 2]]),                  # không trùng
    ([[2, 2]], [[1, 1]], [[1, 1], [2, 2]]),                  # thứ tự ngược
    ([[1000, 1000]], [[1000, 1000]], [[1000, 2000]]),       # biên trên
    ([[1, 1]], [[1000, 1]], [[1, 1], [1000, 1]]),            # hai đầu mút
    ([[3, 1], [2, 1], [1, 1]], [[1, 1], [2, 1], [3, 1]], [[1, 2], [2, 2], [3, 2]]),
    ([[10, 1], [20, 2]], [[5, 3], [15, 4], [25, 5]], [[5, 3], [10, 1], [15, 4], [20, 2], [25, 5]]),
]
for a, b, expected in fixed:
    for sol in solvers:
        got = sol.mergeSimilarItems([x[:] for x in a], [x[:] for x in b])
        assert got == expected, (type(sol).__name__, a, b, got, expected)
print("test cố định: OK")

def reference(a, b):
    d = {}
    for v, w in a:
        d[v] = d.get(v, 0) + w
    for v, w in b:
        d[v] = d.get(v, 0) + w
    out = []
    for v in range(1, 1001):
        if v in d:
            out.append([v, d[v]])
    return out

random.seed(2363)
for _ in range(500):
    na, nb = random.randint(1, 15), random.randint(1, 15)
    hi = random.choice([5, 20, 1000])
    va = random.sample(range(1, hi + 1), min(na, hi))
    vb = random.sample(range(1, hi + 1), min(nb, hi))
    a = [[v, random.randint(1, 1000)] for v in va]
    b = [[v, random.randint(1, 1000)] for v in vb]
    expected = reference(a, b)
    for sol in solvers:
        got = sol.mergeSimilarItems([x[:] for x in a], [x[:] for x in b])
        assert got == expected, (type(sol).__name__, a, b)
print("500 test ngẫu nhiên: OK")

# kích thước tối đa: 1000 + 1000 phần tử, giá trị 1..1000 cho cả hai
big_a = [[v, v] for v in range(1000, 0, -1)]
big_b = [[v, 1] for v in range(1, 1001)]
big_expected = [[v, v + 1] for v in range(1, 1001)]
for sol in solvers:
    assert sol.mergeSimilarItems([x[:] for x in big_a], [x[:] for x in big_b]) == big_expected, type(sol).__name__
print("kích thước tối đa: OK")

test cố định: OK
500 test ngẫu nhiên: OK


kích thước tối đa: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên sắp xếp kết quả

Đề yêu cầu kết quả **tăng dần theo `value`**; từ điển giữ thứ tự chèn, không phải thứ tự giá trị.

In [21]:
def wrong1(items1, items2):
    total = {}
    for v, w in items1 + items2:
        total[v] = total.get(v, 0) + w
    return [[v, w] for v, w in total.items()]          # thiếu sorted

print("sai :", wrong1([[1, 1], [4, 5], [3, 8]], [[3, 1], [1, 5]]), "(thứ tự 1,4,3; đáng lẽ 1,3,4)")
print("đúng:", Solution().mergeSimilarItems([[1, 1], [4, 5], [3, 8]], [[3, 1], [1, 5]]))

sai : [[1, 6], [4, 5], [3, 9]] (thứ tự 1,4,3; đáng lẽ 1,3,4)
đúng: [[1, 6], [3, 9], [4, 5]]


### Lỗi 2: Ghi đè thay vì cộng dồn

`total[v] = w` làm mất trọng lượng của danh sách đầu.

In [22]:
def wrong2(items1, items2):
    total = {}
    for v, w in items1 + items2:
        total[v] = w                                    # ghi đè
    return [[v, total[v]] for v in sorted(total)]

print("sai :", wrong2([[1, 1], [4, 5], [3, 8]], [[3, 1], [1, 5]]), "(đáng lẽ [[1,6],[3,9],[4,5]])")
print("đúng:", Solution().mergeSimilarItems([[1, 1], [4, 5], [3, 8]], [[3, 1], [1, 5]]))

sai : [[1, 5], [3, 1], [4, 5]] (đáng lẽ [[1,6],[3,9],[4,5]])
đúng: [[1, 6], [3, 9], [4, 5]]


### Lỗi 3: Chỉ xét các `value` có ở cả hai danh sách

Quên các giá trị chỉ xuất hiện ở một danh sách (thiếu ở bên kia nghĩa là trọng lượng `0`, vẫn phải giữ).

In [23]:
def wrong3(items1, items2):
    d1, d2 = dict(items1), dict(items2)
    return [[v, d1[v] + d2[v]] for v in sorted(d1.keys() & d2.keys())]     # giao, mất giá trị lẻ

print("sai :", wrong3([[1, 1], [4, 5], [3, 8]], [[3, 1], [1, 5]]), "(mất [4,5])")
print("đúng:", Solution().mergeSimilarItems([[1, 1], [4, 5], [3, 8]], [[3, 1], [1, 5]]))

sai : [[1, 6], [3, 9]] (mất [4,5])
đúng: [[1, 6], [3, 9], [4, 5]]


### Lỗi 4: `d[v] += w` khi khóa chưa tồn tại

Dùng `d.get(v, 0)`, `Counter` hoặc `defaultdict(int)`.

In [24]:
def wrong4(items1, items2):
    total = {}
    for v, w in items1 + items2:
        total[v] += w                                   # KeyError ở lần đầu
    return sorted(total.items())

try:
    wrong4([[1, 1]], [[1, 2]])
except KeyError as e:
    print("sai -> KeyError:", e)
print("đúng:", Solution().mergeSimilarItems([[1, 1]], [[1, 2]]))

sai -> KeyError: 1
đúng: [[1, 3]]


### Lỗi 5: Trả về `tuple` thay vì `list` `[value, weight]`

`sorted(total.items())` cho danh sách các **tuple** `(v, w)`. LeetCode so sánh theo giá trị nên đôi khi vẫn qua, nhưng về mặt kiểu là sai: `(1, 6) != [1, 6]` trong Python.

In [25]:
def wrong5(items1, items2):
    total = {}
    for v, w in items1 + items2:
        total[v] = total.get(v, 0) + w
    return sorted(total.items())

got = wrong5([[1, 1]], [[1, 2]])
print("sai :", got, "| bằng [[1, 3]]?", got == [[1, 3]])
print("đúng:", Solution().mergeSimilarItems([[1, 1]], [[1, 2]]), "| bằng [[1, 3]]?", Solution().mergeSimilarItems([[1, 1]], [[1, 2]]) == [[1, 3]])

sai : [(1, 3)] | bằng [[1, 3]]? False
đúng: [[1, 3]] | bằng [[1, 3]]? True


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Gộp hai danh sách theo khóa, cộng giá trị" → **`dict`/`Counter`**: `d[k] = d.get(k, 0) + v`.
- "Trả về theo thứ tự khóa" → `sorted(d)` hoặc **mảng đếm theo chỉ số** nếu miền nhỏ (`≤ 1000`).
- Hai danh sách **đã sắp xếp theo khóa** → **hai con trỏ** (như bước trộn của merge sort).
- Đừng nhầm `value` (định danh món đồ) với số lượng; `weight` mới là thứ được cộng.
- Dùng `[[k, v] for ...]` để ra `list`, không dùng `tuple`.
- Cùng họ: LeetCode 88 Merge Sorted Array, 21 Merge Two Sorted Lists, 350 Intersection of Two Arrays II.

## 15. Biến thể của bài

1. Bản **đầy đủ** (không đảm bảo `value` duy nhất trong mỗi mảng): cách `dict` vẫn đúng.
2. Gộp **`k` mảng** thay vì hai (dùng `Counter` cho tất cả hoặc `heapq.merge`).
3. **Trừ** thay vì cộng (hiệu hai kho hàng), bỏ các mục bằng `0`.
4. LeetCode 2570 Merge Two 2D Arrays by Summing Values: gần như cùng bài (tổng theo `id`).
5. Trả về theo thứ tự **giảm dần của `weight`** (sắp xếp phụ thuộc giá trị tổng).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Quét lại mỗi `value` | không dùng cấu trúc phụ | `O((n+m)²)` | `O(n+m)` | chỉ để hiểu đề |
| 2. `dict` + `sorted` | cộng dồn theo khóa | `O(N log N)` | `O(N)` | **khuyến nghị khi nộp** |
| 3. `Counter` | `c[v] += w` | `O(N log N)` | `O(N)` | không lo `KeyError` |
| 4. Mảng 1001 ô | chỉ số = `value` | `O(N + V)` | `O(V)` | miền giá trị nhỏ, không cần sort |
| 5. Hai con trỏ | trộn hai danh sách đã sắp | `O(N log N)` | `O(N)` | bài đã có dữ liệu sắp xếp |
| 6. Nối + `groupby` | gom khóa liền kề | `O(N log N)` | `O(N)` | khóa có thể lặp trong một mảng |

(`N = n + m`.)

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** `d = {}; d[3] = d.get(3, 0) + 5; d[3] = d.get(3, 0) + 2; print(d, sorted({4: 1, 2: 9}.items()))` in ra gì?

**Bài 2 (tính tay).** `items1 = [[2,5],[1,1]]`, `items2 = [[1,2],[3,4],[2,1]]`. Kết quả gộp?

**Bài 3 (điền TODO).** Hoàn thành hàm **trừ** hai danh sách (`items1 - items2`), giữ các `value` có hiệu khác `0`, sắp xếp tăng dần.

In [26]:
def subtract_items(items1, items2):
    # TODO: total[v] += w cho items1, total[v] -= w cho items2; bỏ các hiệu bằng 0
    pass

# chưa gọi hàm ở đây để notebook chạy không lỗi

**Bài 4 (tự viết lại).** Viết `merge_many(list_of_items)` gộp **nhiều** danh sách `[value, weight]` (mỗi danh sách là một `items`), trả về giống bài gốc.

In [27]:
def merge_many(list_of_items):
    # TODO: Counter hoặc dict cộng dồn qua mọi danh sách, rồi sắp xếp
    pass

**Bài 5 (mini challenge).** Viết `top_heaviest(items1, items2, k)` trả về `k` cặp `[value, total_weight]` có tổng trọng lượng **lớn nhất** (nếu hòa, `value` nhỏ hơn trước).

In [28]:
def top_heaviest(items1, items2, k):
    # TODO: gộp rồi sorted với key=lambda t: (-t[1], t[0]), lấy k phần tử đầu
    pass

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `d = {3: 7}`; `sorted` trả `[(2, 9), (4, 1)]` (tuple). Bài 2: `value 1: 1+2=3`; `value 2: 5+1=6`; `value 3: 4`.
- **Mức 2 (vừa):** Bài 3: `Counter()`; cộng items1, trừ items2; lọc `total[v] != 0`. Bài 4: nối mọi danh sách rồi áp dụng cách `dict` cộng dồn.
- **Mức 3 (sâu):** Bài 5: `merged = Solution().mergeSimilarItems(...)` rồi `sorted(merged, key=lambda t: (-t[1], t[0]))[:k]`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [29]:
from collections import Counter

# Bài 1 và 2
d = {}
d[3] = d.get(3, 0) + 5
d[3] = d.get(3, 0) + 2
assert d == {3: 7} and sorted({4: 1, 2: 9}.items()) == [(2, 9), (4, 1)]
assert Solution().mergeSimilarItems([[2, 5], [1, 1]], [[1, 2], [3, 4], [2, 1]]) == [[1, 3], [2, 6], [3, 4]]

# Bài 3
def subtract_items(items1, items2):
    total = Counter()
    for v, w in items1:
        total[v] += w
    for v, w in items2:
        total[v] -= w
    return [[v, total[v]] for v in sorted(total) if total[v] != 0]

# Bài 4
def merge_many(list_of_items):
    total = Counter()
    for items in list_of_items:
        for v, w in items:
            total[v] += w
    return [[v, total[v]] for v in sorted(total)]

# Bài 5
def top_heaviest(items1, items2, k):
    merged = Solution().mergeSimilarItems(items1, items2)
    return sorted(merged, key=lambda t: (-t[1], t[0]))[:k]

assert subtract_items([[1, 5], [2, 3]], [[1, 5], [3, 1]]) == [[2, 3], [3, -1]]
assert subtract_items([[1, 1]], [[1, 1]]) == []
assert merge_many([[[1, 1]], [[1, 2], [2, 3]], [[2, 4]]]) == [[1, 3], [2, 7]]
assert merge_many([[[5, 5]]]) == [[5, 5]]
assert top_heaviest([[1, 1], [4, 5], [3, 8]], [[3, 1], [1, 5]], 2) == [[3, 9], [1, 6]]
assert top_heaviest([[1, 5], [2, 5]], [], 1) == [[1, 5]]
assert top_heaviest([[1, 2]], [[2, 2]], 5) == [[1, 2], [2, 2]]
print("tất cả bài tập OK")

tất cả bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (cộng dồn theo khóa rồi sắp xếp)

```python
class Solution:
    def mergeSimilarItems(self, items1: List[List[int]], items2: List[List[int]]) -> List[List[int]]:
        total = {}
        for value, weight in items1 + items2:
            total[value] = total.get(value, 0) + weight
        return [[value, total[value]] for value in sorted(total)]
```

**Checklist**

- [ ] **Cộng dồn** (`get(..., 0) + w`), không ghi đè?
- [ ] Giữ cả các `value` chỉ có ở **một** danh sách?
- [ ] Kết quả **sắp xếp tăng dần theo `value`**?
- [ ] Phần tử kết quả là `list` `[value, weight]`, không phải `tuple`?
- [ ] Không bị `KeyError` khi cộng (`get`, `Counter` hoặc `defaultdict`)?
- [ ] Đã thử hai danh sách thứ tự ngược, trùng hoàn toàn và không trùng giá trị nào?
- [ ] Signature đúng `mergeSimilarItems(self, items1: List[List[int]], items2: List[List[int]]) -> List[List[int]]` và có `from typing import List`?